# Three histories, interleaved

The Rust prototype, the Rust port that replaced it, and the C library both were ported from,
drawn as commits per day over the months when all three were in play.
The point of the figure is the conversation between the lanes: each time the port demanded
something the C side could not supply, the C side changed, and the marked events say where.

Commits come straight from local clones with `git log --no-merges`, not from the line-count
CSVs, because the C side's activity is invisible on a first-parent walk: this window holds
431 commits in the C lane, where a first-parent walk of `master` sees 34, only 11 of which
are not merges.

| Lane | Repository | Ref | Why this ref |
|---|---|---|---|
| prototype | `starlink-astrs-proto` | `main` | the whole prototype |
| port | `starlink-astrs` | `tickets/DM-54793` | this repository, on its development branch |
| C library | `ast` | `u/timj/misc-bugs` | `master` plus the fixes still in flight |

Commits authored by embray are dropped from the C lane: that work was unrelated to the port,
so it would read as noise in a figure about the port's influence.
Everything outside `u/timj/misc-bugs` in this window is a pre-squash backup branch, which
would double-count the same work.

Days are the author's own calendar date, taken from the first ten characters of the commit
stamp rather than converted to UTC, so that "the C side answered the same day" means what it
says even for a commit made late in the evening.

The commit list is cached to `data/commit-history.csv` beside this notebook on first run,
and that file is committed.
Once the development branch is squashed away it is the only remaining record of the port's
1,166 commits, so calling `commit_history(refresh=True)` after that point would silently
shrink the figure, and re-collecting is not a way to recover it.

The two sibling repositories are read from the local clone paths in `TRACKS`; adjust them if
yours live elsewhere.
Paths are relative to this directory, so run the notebook from `docs/reports`.


In [ ]:
%matplotlib widget

In [ ]:
import subprocess
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# One lane per repository, drawn top to bottom in this order. Colours are the
# first three slots of a categorical palette validated for colour-blind
# separation; each lane also carries its name, so identity never rests on
# colour alone.
TRACKS = (
    {
        "track": "prototype",
        "label": "starlink-astrs-proto  (Rust prototype)",
        "repo": "~/work/lsst/starlink-astrs-proto",
        "ref": "main",
        "exclude": (),
        "color": "#2a78d6",
    },
    {
        "track": "port",
        "label": "starlink-astrs  (Rust port)",
        "repo": "../..",
        "ref": "tickets/DM-54793",
        "exclude": (),
        "color": "#eb6834",
    },
    {
        "track": "ast",
        "label": "ast  (C library)",
        "repo": "~/work/ast",
        "ref": "u/timj/misc-bugs",
        "exclude": ("embray",),
        "color": "#1baf7a",
    },
)

# Moments where influence crossed between the repositories, each one anchored
# to a commit that can be found in the history rather than to a narrative.
# The lane names the repository the event happened in; the height and
# alignment are hand-chosen so that neighbouring labels do not collide.
EVENTS = (
    ("2026-04-27", "port", "port begins,\npinned to C", 0.80, "center"),
    ("2026-05-08", "ast", "C adds a fixture\ngenerator for the port", 0.80, "center"),
    ("2026-06-04", "ast", "C retires the\nC99/C11 digit hazard", 0.80, "center"),
    ("2026-06-18", "ast", "first port-found\ndefects fixed in C", 0.46, "center"),
    ("2026-06-27", "ast", "C builds the\ntransform oracle", 0.80, "center"),
    (
        "2026-08-08",
        "ast",
        "KeyMap defects filed\nand fixed the same day",
        0.80,
        "center",
    ),
)

# Beside this notebook, and committed: see the note at the top.
CACHE = Path("data/commit-history.csv")

START, END = "2026-03-01", "2026-08-23"

INK = "#1a1a19"
MUTED = "#6b6a63"
GUIDE = "#a9a8a0"
GRID = "#e6e5e1"
SURFACE = "#ffffff"

In [ ]:
def collect(repo, ref, exclude=()):
    """Read the non-merge commits reachable from a ref.

    Parameters
    ----------
    repo : `str`
        Path to a local clone, tilde allowed.
    ref : `str`
        Branch or commit to walk back from.
    exclude : `tuple` [ `str` ], optional
        Substrings of author email addresses whose commits to drop.

    Returns
    -------
    commits : `pandas.DataFrame`
        One row per commit, with ``commit``, ``date`` and ``author`` columns.
        The date is the author date in strict ISO form, offset included.
    """
    text = subprocess.run(
        [
            "git",
            "-C",
            str(Path(repo).expanduser()),
            "log",
            "--no-merges",
            "--format=%H%x09%aI%x09%an%x09%ae",
            ref,
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    rows = []
    for line in text.splitlines():
        commit, date, author, email = line.split("\t")
        if any(pattern in email for pattern in exclude):
            continue
        rows.append((commit, date, author))
    return pd.DataFrame(rows, columns=["commit", "date", "author"])


def commit_history(refresh=False):
    """Load the cached commit list for all three lanes.

    Parameters
    ----------
    refresh : `bool`, optional
        Re-read the repositories and overwrite the cache. A lane whose branch
        no longer exists cannot be re-read, so this shrinks the figure once
        the port's development branch has been squashed away.

    Returns
    -------
    history : `pandas.DataFrame`
        The rows of `collect` for every lane, with a ``track`` column naming
        the lane.
    """
    if CACHE.exists() and not refresh:
        print(f"read {CACHE}")
        return pd.read_csv(CACHE)
    frames = []
    for spec in TRACKS:
        frame = collect(spec["repo"], spec["ref"], spec["exclude"])
        frame.insert(0, "track", spec["track"])
        frames.append(frame)
    history = pd.concat(frames, ignore_index=True)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    history.to_csv(CACHE, index=False)
    print(f"wrote {CACHE}")
    return history

In [ ]:
history = commit_history()

# The first ten characters of an ISO commit stamp are the author's own
# calendar date, which is the sense in which the C side answered the same day.
history["day"] = pd.to_datetime(history["date"].str.slice(0, 10))
window = history[history["day"].between(START, END)]

# Reindexing onto every day in the window turns the quiet stretches into
# explicit zeros, so a gap in the bars is a gap in the work.
days = pd.date_range(START, END, freq="D")
daily = {
    spec["track"]: (
        window[window["track"] == spec["track"]]
        .groupby("day")
        .size()
        .reindex(days, fill_value=0)
    )
    for spec in TRACKS
}

for spec in TRACKS:
    counts = daily[spec["track"]]
    print(
        f"{spec['track']:>10}: {int(counts.sum()):>5} commits, "
        f"{int((counts > 0).sum()):>3} active days, {int(counts.max()):>3} at most in a day"
    )

In [ ]:
fig, lanes = plt.subplots(
    len(TRACKS), 1, sharex=True, figsize=(11, 6.4), gridspec_kw={"hspace": 0.5}
)
axis_of = dict(zip([spec["track"] for spec in TRACKS], lanes, strict=True))

# One shared vertical scale across the lanes. Per-lane scales would make the
# quieter repository look as busy as the louder one.
ceiling = max(counts.max() for counts in daily.values())

for ax, spec in zip(lanes, TRACKS, strict=True):
    counts = daily[spec["track"]]
    ax.bar(
        counts.index,
        counts.to_numpy(),
        width=0.85,
        color=spec["color"],
        linewidth=0,
        zorder=3,
    )
    ax.set_ylim(0, ceiling * 1.06)
    ax.set_yticks([0, 50, 100])
    ax.grid(axis="y", color=GRID, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, length=3, labelsize=9)
    ax.plot(
        [0.0],
        [1.14],
        marker="s",
        markersize=6,
        color=spec["color"],
        transform=ax.transAxes,
        clip_on=False,
    )
    ax.set_title(
        f"    {spec['label']}   ·   {int(counts.sum()):,} commits, {int((counts > 0).sum())} active days",
        loc="left",
        color=INK,
        fontsize=9.5,
        pad=8,
    )

for date, track, text, height, align in EVENTS:
    when = pd.Timestamp(date)
    # The guide line crosses every lane, because the interesting part is what
    # the other two repositories were doing on that date.
    for ax in lanes:
        ax.axvline(when, color=GUIDE, linewidth=0.7, linestyle=(0, (3, 3)), zorder=1)
    # A surface-coloured halo keeps the guide line pointing at the label
    # rather than striking through it.
    axis_of[track].text(
        when,
        height * ceiling,
        text,
        ha=align,
        va="bottom",
        fontsize=7.5,
        color=MUTED,
        linespacing=1.35,
        zorder=4,
        bbox={"facecolor": SURFACE, "edgecolor": "none", "pad": 1.5},
    )

lanes[-1].xaxis.set_major_locator(mdates.MonthLocator())
lanes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
lanes[-1].xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
lanes[-1].set_xlim(pd.Timestamp(START), pd.Timestamp(END))
lanes[-1].set_xlabel("2026", color=MUTED, fontsize=9.5)
fig.supylabel("commits per day", color=MUTED, fontsize=9.5, x=0.045)
fig.suptitle(
    "Three repositories, one conversation: commits per day",
    color=INK,
    fontsize=12,
    x=0.045,
    ha="left",
    y=1.02,
)

In [ ]:
# The title belongs to the figure here and to the caption in a document, so
# drop it on the way out.
fig.suptitle("")
fig.savefig("starlink-port-interleave.svg", bbox_inches="tight")